# Broadcast Join
- A broadcast join in PySpark optimizes join operations by sending a smaller DataFrame to all worker nodes, allowing for local joins and minimizing data shuffling.

### 💡 Note
- In order to use Broadcast Join, the smaller DataFrame should be able to fit in `Spark Drivers` and `Executors memory`. 
- If the DataFrame can’t fit in memory you will be getting `out-of-memory` errors.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.master("local[5]") \
    .appName("broadcastJoin") \
    .getOrCreate()

spark

In [2]:
customer_data = [(1, 'manish', 'patna', '30-05-2022'),
(2, 'vikash', 'kolkata', '12-03-2023'),
(3, 'nikita', 'delhi', '25-06-2023'),
(4, 'rahul', 'ranchi', '24-03-2023'),
(5, 'mahesh', 'jaipur', '22-03-2023'),
(6, 'prantoshi', 'kolkata', '18-10-2022'),
(7, 'raman', 'patna', '30-12-2022'),
(8, 'prakash', 'ranchi', '24-02-2023'),
(9, 'ragini', 'kolkata', '03-03-2023'),
(10, 'raushan', 'jaipur', '05-02-2023')]

customer_schema = ['customer_id', 'customer_name', 'address', 'date_of_join']

customer_df = spark.createDataFrame(data=customer_data, schema=customer_schema)
customer_df.createOrReplaceTempView("customer_tbl")

In [3]:
sales_data = [(1, 22, 10, "01-06-2022"),
(1, 27, 5, "03-02-2023"),
(2, 5, 3, "01-06-2023"),
(5, 12, 1, "22-03-2023"),
(7, 22, 4, "03-02-2023"),
(9, 5, 6, "03-03-2023"),
(2, 1, 12, "15-06-2023"),
(1, 56, 2, "25-06-2023"),
(5, 12, 5, "15-04-2023"),
(11, 12, 76, "12-03-2023")]

sales_schema = ['customer_id', 'product_id', 'quantity', 'date_of_purchase']

sales_df = spark.createDataFrame(data=sales_data, schema=sales_schema)
sales_df.createOrReplaceTempView("sales_tbl")

In [4]:
# Prefer join operation in spark SQL is sortMerge join
sort_merge_df = customer_df.join(sales_df, customer_df['customer_id'] == sales_df['customer_id'], 'inner')

sort_merge_df.show()

+-----------+-------------+-------+------------+-----------+----------+--------+----------------+
|customer_id|customer_name|address|date_of_join|customer_id|product_id|quantity|date_of_purchase|
+-----------+-------------+-------+------------+-----------+----------+--------+----------------+
|          1|       manish|  patna|  30-05-2022|          1|        22|      10|      01-06-2022|
|          1|       manish|  patna|  30-05-2022|          1|        27|       5|      03-02-2023|
|          1|       manish|  patna|  30-05-2022|          1|        56|       2|      25-06-2023|
|          2|       vikash|kolkata|  12-03-2023|          2|         5|       3|      01-06-2023|
|          2|       vikash|kolkata|  12-03-2023|          2|         1|      12|      15-06-2023|
|          5|       mahesh| jaipur|  22-03-2023|          5|        12|       1|      22-03-2023|
|          5|       mahesh| jaipur|  22-03-2023|          5|        12|       5|      15-04-2023|
|          7|       

In [5]:
sort_merge_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [customer_id#0L], [customer_id#4L], Inner
   :- Sort [customer_id#0L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=145]
   :     +- Filter isnotnull(customer_id#0L)
   :        +- Scan ExistingRDD[customer_id#0L,customer_name#1,address#2,date_of_join#3]
   +- Sort [customer_id#4L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#4L, 200), ENSURE_REQUIREMENTS, [plan_id=146]
         +- Filter isnotnull(customer_id#4L)
            +- Scan ExistingRDD[customer_id#4L,product_id#5L,quantity#6L,date_of_purchase#7]




In [6]:
# Broadcast Join
# This is used when one of the table is small and can be broadcasted to all the nodes(executor)
broadcast_df = customer_df.join(broadcast(sales_df), customer_df['customer_id'] == sales_df['customer_id'], 'inner')
broadcast_df.show()
broadcast_df.explain()

+-----------+-------------+-------+------------+-----------+----------+--------+----------------+
|customer_id|customer_name|address|date_of_join|customer_id|product_id|quantity|date_of_purchase|
+-----------+-------------+-------+------------+-----------+----------+--------+----------------+
|          1|       manish|  patna|  30-05-2022|          1|        56|       2|      25-06-2023|
|          1|       manish|  patna|  30-05-2022|          1|        27|       5|      03-02-2023|
|          1|       manish|  patna|  30-05-2022|          1|        22|      10|      01-06-2022|
|          2|       vikash|kolkata|  12-03-2023|          2|         1|      12|      15-06-2023|
|          2|       vikash|kolkata|  12-03-2023|          2|         5|       3|      01-06-2023|
|          5|       mahesh| jaipur|  22-03-2023|          5|        12|       5|      15-04-2023|
|          5|       mahesh| jaipur|  22-03-2023|          5|        12|       1|      22-03-2023|
|          7|       

In [ ]:
# How to get and set the broadcast threshold

print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1000000")  # Set to 1 MB
# print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# if you want to disable broadcast join
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # Disable broadcast join

10485760b
